<a href="https://colab.research.google.com/github/Santosh-S321/Agentic_AI/blob/main/agentic_lab2/LAB_PROGRAM_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Chatbot that Remembers, using LangGraph

In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(api_key= groq_api_key,
               model="llama-3.1-8b-instant", temperature=0)

In [3]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]     # the running conversation
    name: str               # remembered user name
    preferences: str        # remembered preferences

In [4]:
def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    # Retrieve memory
    name = state.get('name', '')
    prefs = state.get('preferences', '')

    # Simple memory extraction rules
    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()

    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    # Context for LLM
    context = f"The user's name is {name}. They like {prefs}."

    # Generate response
    reply = llm.invoke(context + ' Reply to: ' + user_msg).content

    # Update state
    return {
        'messages': state['messages'] + [reply],
        'name': name,
        'preferences': prefs
    }

In [5]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)

builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)

graph = builder.compile()

In [6]:
state = {
    'messages': [],
    'name': '',
    'preferences': ''
}

print("🤖 Chatbot started! Type 'quit' to exit.\n")

while True:
    user = input("You: ")

    if user.lower() == 'quit':
        print("👋 Goodbye!")
        break

    state['messages'].append(user)

    state = graph.invoke(state)

    print("Bot:", state['messages'][-1])

🤖 Chatbot started! Type 'quit' to exit.

You: Arun
Bot: Hello Arun, it's nice to meet you. What do you like?
You: I'm a coding freak interested in Python language
Bot: Nice to meet you, . I'm glad to hear that you're a coding freak and interested in Python language. Python is an amazing language to learn, with a vast range of applications in web development, data analysis, machine learning, and more.

What specifically draws you to Python? Are you looking to build a project, learn data science, or perhaps create a web application?
You: Building a Project
Bot: Building a project sounds like an exciting endeavor. What kind of project are you looking to build, and what's driving your interest in it?
You: quit
👋 Goodbye!
